### What encapsulation means

In [ ]:
class Account:
    def __init__(self, owner, balance):
        self.owner = owner
        self.balance = balance

acc1 = Account("Kavya", 100)

acc1.balance = -5000   # nothing stops this
print("acc1.balance:", acc1.balance)

Encapsulation is the general idea of fixing exactly this: controlling who can read or change an object's internal state, and how they're allowed to do it

#### Public attributes

self.owner, self.balance — no prefix, nothing special — these are public attributes, which is Python's default: freely readable from outside (acc1.balance) and freely writable from outside 

### _var (single underscore)

In [ ]:
class Account:
    def __init__(self, owner, balance):
        self.owner = owner
        self._balance = balance   # single underscore

acc1 = Account("Kavya", 100)

print("acc1._balance:", acc1._balance)

acc1._balance = -5000
print("acc1._balance:", acc1._balance)

In [6]:
class Account:
    def __init__(self, owner, balance):
        self.owner = owner          # public - meant to be used freely from outside
        self._balance = balance     # single underscore - meant as internal, not enforced

acc1 = Account("Kavya", 100)

acc1.owner = "Someone else"    # public - works, as expected
acc1._balance = -5000           # underscore - ALSO works, Python doesn't block it either

1. owner (no underscore, public) = "this is part of the intended, outside-facing interface — go ahead and use it directly."
2. _balance (underscore) = "this is an internal implementation detail — I didn't intend for you to touch this from outside, please go through a proper method instead, even though I have no way to actually stop you."

So _var isn't "means public" — it's "looks and behaves exactly like public, mechanically, but is a deliberate signal meaning the opposite: 'treat this as private, on the honor system.'" It's a label about intent, not a technical restriction 

### Concept 4: __var (double underscore)



In [ ]:
class Account:
    def __init__(self, owner, balance):
        self.owner = owner
        self.__balance = balance   # double underscore

acc1 = Account("Kavya", 100)

print(acc1.__balance)

In [9]:
print("acc1.__dict__:", acc1.__dict__)
print("acc1._Account__balance:", acc1._Account__balance)

acc1.__dict__: {'owner': 'Kavya', '_Account__balance': 100}
acc1._Account__balance: 100


 the moment you wrote self.__balance = balance inside the Account class, Python silently rewrote it to self._Account__balance = balance — the class name got stitched onto the front. That's why acc1.__balance fails: that key genuinely doesn't exist. But acc1._Account__balance — the real, mangled name — works completely fine, no restriction at all.

### Why name mangling exists

In [12]:
class Account:
    def __init__(self, owner):
        print("  Account.__init__ running: setting self.owner")
        self.owner = owner

class SavingsAccount(Account):
    def __init__(self, owner):
        print("SavingsAccount.__init__ started")
        super().__init__(owner)     # runs Account's __init__ on this same self
        print("Back in SavingsAccount.__init__, self.owner is now:", self.owner)

s = Account("Kavya")
print("s.__dict__:", s.__dict__)
s2 = SavingsAccount("Kavya")
print("s2.__dict__:", s2.__dict__)


  Account.__init__ running: setting self.owner
s.__dict__: {'owner': 'Kavya'}
SavingsAccount.__init__ started
  Account.__init__ running: setting self.owner
Back in SavingsAccount.__init__, self.owner is now: Kavya
s2.__dict__: {'owner': 'Kavya'}


SavingsAccount.__init__ starts first, then Account.__init__ runs in the middle, then control comes back to SavingsAccount.__init__ to finish. super().__init__(owner) literally means: "go run the parent class's (Account's) __init__ method, right now, on this same self — then come back and keep going." SavingsAccount doesn't want to rewrite self.owner = owner itself — it just borrows Account's existing setup logic for that part, then adds whatever extra setup it needs afterward.

In [13]:
class Account:
    def __init__(self, owner):
        self.owner = owner
        self.__id = "account-internal-id"

class SavingsAccount(Account):
    def __init__(self, owner):
        super().__init__(owner)
        self.__id = "savings-internal-id"

s = SavingsAccount("Kavya")
print("s.__dict__:", s.__dict__)

s.__dict__: {'owner': 'Kavya', '_Account__id': 'account-internal-id', '_SavingsAccount__id': 'savings-internal-id'}


### Manual getter/setter methods

In [16]:
class Account:
    def __init__(self, owner, balance):
        self.owner = owner
        self._balance = balance

    def get_balance(self):
        return self._balance

    def set_balance(self, value):
        if value < 0:
            print("  Rejected: balance cannot be negative")
            return
        self._balance = value

acc1 = Account("Kavya", 100)

print("get_balance():", acc1.get_balance())
acc1.set_balance(-5000)
print("after set_balance(-5000):", acc1.get_balance())
acc1.set_balance(5000)
print("after set_balance(500):", acc1.get_balance())

get_balance(): 100
after set_balance(-5000): 5000
after set_balance(500): 500


This is the traditional, verbose way to get real encapsulation: rename the attribute with a leading underscore (_balance, signaling "don't touch this directly"), then write explicit get_x()/set_x() methods as the only sanctioned door in and out. It genuinely works — but notice the cost: every single attribute you want to protect needs its own pair of methods, and callers now have to write acc1.set_balance(500) instead of the natural acc1.balance = 500, everywhere, forever. That verbosity is exactly what the Python community considers ugly enough to solve differently.

In [17]:
class Account:
    def __init__(self, owner, balance):
        self.owner = owner
        self._balance = balance

    @property
    def balance(self):
        return self._balance

    @balance.setter
    def balance(self, value):
        if value < 0:
            print("  Rejected: balance cannot be negative")
            return
        self._balance = value

acc1 = Account("Kavya", 100)

print("acc1.balance:", acc1.balance)
acc1.balance = -5000
print("acc1.balance:", acc1.balance)
acc1.balance = 500
print("acc1.balance:", acc1.balance)

acc1.balance: 100
  Rejected: balance cannot be negative
acc1.balance: 100
acc1.balance: 500


#### @property gets you the clean, natural syntax of a public attribute, plus all the control of manual getters/setters, at the same time — without the caller ever needing to know a method is involved. That resolves the exact tradeoff from the last two concepts: Python doesn't force you to choose between "nice syntax" and "real control" the way plain getters/setters do.

### Name mangling also protects methods, not just attributes

So far __balance / __id showed mangling protecting a *stored value* from being overwritten by a subclass. The same mechanism protects a *method call* from being hijacked by a subclass that overrides that method.

In [ ]:
# BROKEN version — __init__ calls the public update() directly

class Mapping:
    def __init__(self, iterable):
        self.items_list = []
        self.update(iterable)          # calls PUBLIC update

    def update(self, iterable):
        for item in iterable:
            self.items_list.append(item)

class MappingSubclass(Mapping):
    def update(self, keys, values):    # overrides update with a DIFFERENT signature
        for item in zip(keys, values):
            self.items_list.append(item)

m = MappingSubclass(["a", "b"])   # crashes: TypeError, missing 'values'

MappingSubclass has no __init__, so creating it runs the inherited Mapping.__init__. That does self.update(iterable) — normal method lookup starts at the object's actual class (MappingSubclass), finds MappingSubclass.update first, and stops there (closest class wins). But MappingSubclass.update needs two arguments (keys, values), and it only got one (iterable) → crash.

Mapping's own constructor got hijacked by a subclass's legitimate override, just because both used the same public name.

In [ ]:
# FIXED version — __init__ calls the mangled __update() instead

class Mapping:
    def __init__(self, iterable):
        self.items_list = []
        self.__update(iterable)        # calls _Mapping__update, not public update

    def update(self, iterable):
        for item in iterable:
            self.items_list.append(item)

    __update = update                  # gives that same function a private nickname

class MappingSubclass(Mapping):
    def update(self, keys, values):
        for item in zip(keys, values):
            self.items_list.append(item)

m = MappingSubclass(["a", "b"])
print("m.items_list after __init__:", m.items_list)

m.update(["k1", "k2"], [1, 2])         # public update() still runs the CHILD's version
print("m.items_list after m.update(...):", m.items_list)

__update = update, written inside Mapping's own class body, gets mangled to _Mapping__update — a second, private label pointing at the exact same function as update. MappingSubclass never defines anything called _Mapping__update, so when Mapping.__init__ calls self.__update(iterable), lookup doesn't find it on MappingSubclass, walks up, and finds Mapping's original one-argument version. No crash.

Calling m.update(["k1","k2"], [1,2]) directly still goes through the public name and correctly runs MappingSubclass's own two-argument version — overriding still works exactly as intended for outside callers.

Important distinction — two different directions:
- Child wants to call the parent's version on purpose → use super().update(...). Normal, everyday, nothing to do with mangling.
- Parent wants to guarantee it always calls its OWN version, immune to being overridden by any child → parent uses self.__update(...) (mangled). This is the opposite direction from super() — it's the base class protecting its own internal call from being hijacked by a legitimate subclass override, not a subclass reaching up to the parent.

Child wants to call parent's method on purpose → super().update(...). Right, unchanged.

Parent wants a guaranteed private call to its own version, immune to whatever the child does with the public name update → self.__update(...). But notice: this is not "parent forbids overriding" or "only kicks in when arguments differ." Mangling doesn't care about signatures at all, and it doesn't block overriding in any way, ever.

#### What "doesn't want children to override" would actually mean — and mangling isn't that

#### There's no way in Python to actually prevent a subclass from overriding a public method. update stays 100% override-able no matter what the parent does. Mangling doesn't touch that. What mangling gives the parent is a second, separate name (_Mapping__update) that the child simply doesn't know about and doesn't touch — so that specific internal call stays pointed at the parent's own code, regardless of whether the child overrode the public update or not, and regardless of whether the child's version happens to have a matching signature or a completely different one. It protects the call unconditionally — not "only when signatures mismatch."


"What if the child wants to actually override?"
It can, freely, no restriction at all — that's exactly what MappingSubclass.update(self, keys, values) already does in the fixed version. Run it and:

m.update(["k1","k2"], [1,2]) → uses the public name → correctly runs the child's overridden version. Overriding works perfectly, no interference.
Meanwhile Mapping.__init__'s internal self.__update(iterable) → uses the mangled name → still runs the parent's original version, untouched by the override.

In [ ]:
class Mapping:
    def __init__(self, iterable):
        self.__update(iterable)     # -> self._Mapping__update(iterable)

    def update(self, iterable):
        self.items_list = []
        for item in iterable:
            self.items_list.append(item)

    __update = update


In [ ]:
m = Mapping([1, 2, 3])
print(m.items_list)   # [1, 2, 3]


 m = Mapping([1, 2, 3])
m.update([4, 5])          # perfectly normal, public method call
print(m.items_list)       # [4, 5]  (since update() resets items_list = [] each time)


### What you can't/shouldn't do

m.__update([1,2,3])          # AttributeError — that literal name was never created
m._Mapping__update([1,2,3])  # works, but you're not supposed to reach in like this
